# Phase 0: データ検証（Go/No-Go ゲート）

`docs/requirements.md` 3章、`CLAUDE.md` の制約に従う。

- データ: `data/raw/animelist.csv`（1億900万行, user_id / anime_id / rating / watching_status / watched_episodes）
- サンプリング方針: `nrows` による先頭読みは user_id の若い層に偏るため使用しない。**user_id をランダム抽出してから該当行を読む**（2パス方式）。目標 2〜5万ユーザー。
- `rating=0` は「未評価」であり最低評価ではない（本ノートブックでは rating 自体は Phase 0 の判定に使用しないため直接の影響はないが、以降のフェーズのために明記する）。


In [1]:
import time
import numpy as np
import pandas as pd

RAW_DIR = "../data/raw"
ANIMELIST_PATH = f"{RAW_DIR}/animelist.csv"
ANIME_PATH = f"{RAW_DIR}/anime.csv"

SEED = 42
N_USERS_TARGET = 30_000
CHUNKSIZE = 10_000_000

STATUS_NAMES = {0: "不明", 1: "視聴中", 2: "完走", 3: "保留", 4: "離脱", 6: "視聴予定"}

pd.set_option("display.float_format", lambda x: f"{x:,.3f}")


## 1. ユーザーIDのランダムサンプリング（第1パス）

`user_id` 列のみを全件スキャンし、ユニークな user_id 一覧を取得する。第2パスの対象を決めるための最小限の読み込み（列を絞っているため高速）。

In [2]:
t0 = time.time()
unique_chunks = []
n_rows_seen = 0
for chunk in pd.read_csv(
    ANIMELIST_PATH,
    usecols=["user_id"],
    dtype={"user_id": "int32"},
    chunksize=CHUNKSIZE,
):
    unique_chunks.append(chunk["user_id"].unique())
    n_rows_seen += len(chunk)

unique_users = np.unique(np.concatenate(unique_chunks))
print(f"rows scanned: {n_rows_seen:,}")
print(f"unique users (population): {len(unique_users):,}")
print(f"pass1 elapsed: {time.time()-t0:.1f}s")


rows scanned: 109,224,747
unique users (population): 325,770
pass1 elapsed: 8.0s


In [3]:
rng = np.random.default_rng(SEED)
n_sample = min(N_USERS_TARGET, len(unique_users))
sampled_users = rng.choice(unique_users, size=n_sample, replace=False)
sampled_users_set = set(sampled_users.tolist())
print(f"sampled users: {n_sample:,} (of {len(unique_users):,} total, {n_sample/len(unique_users)*100:.1f}%)")


sampled users: 30,000 (of 325,770 total, 9.2%)


## 2. サンプルユーザーの全行を抽出（第2パス）

抽出した user_id 集合に該当する行のみを、全列を読みながらチャンク単位でフィルタして集約する。

In [4]:
t1 = time.time()
dtypes = {
    "user_id": "int32",
    "anime_id": "int32",
    "rating": "int8",
    "watching_status": "int8",
    "watched_episodes": "int32",
}
parts = []
n_rows_seen2 = 0
for chunk in pd.read_csv(ANIMELIST_PATH, dtype=dtypes, chunksize=CHUNKSIZE):
    n_rows_seen2 += len(chunk)
    mask = chunk["user_id"].isin(sampled_users_set)
    if mask.any():
        parts.append(chunk.loc[mask])

df = pd.concat(parts, ignore_index=True)
del parts

n_rows = len(df)
n_users = df["user_id"].nunique()
print(f"rows scanned (full data): {n_rows_seen2:,}")
print(f"sample rows: {n_rows:,} / sample users: {n_users:,}")
print(f"pass2 elapsed: {time.time()-t1:.1f}s | total elapsed: {time.time()-t0:.1f}s")
df.head()


rows scanned (full data): 109,224,747
sample rows: 10,038,349 / sample users: 30,000
pass2 elapsed: 13.0s | total elapsed: 21.0s


,user_id,anime_id,rating,watching_status,watched_episodes
0,0,67,9,1,1
1,0,6702,7,1,4
2,0,242,10,1,4
3,0,4898,0,1,1
4,0,21,10,1,0


## 3. `watching_status` 分布の検証

要件定義書は「必ず実データの分布で検証すること」と定める。ランダムサンプリングされた3万ユーザー・約1000万行に基づく分布を算出する。

In [5]:
status_counts = df["watching_status"].value_counts().sort_index()
status_pct = (status_counts / n_rows * 100).round(3)
dist_table = pd.DataFrame({"count": status_counts, "pct(%)": status_pct})
dist_table.index = [f"{i} ({STATUS_NAMES.get(i, '未知の値')})" for i in dist_table.index]
dist_table


,count,pct(%)
0 (不明),3,0.000
1 (視聴中),478018,4.762
2 (完走),6306027,62.819
3 (保留),342226,3.409
4 (離脱),404260,4.027
5 (未知の値),2,0.000
6 (視聴予定),2507813,24.982


既知の値（0, 1, 2, 3, 4, 6）以外のステータス値が含まれる場合は、データ品質上の異常として扱う（下セルで検出）。

In [6]:
known_statuses = {0, 1, 2, 3, 4, 6}
unknown_status_mask = ~df["watching_status"].isin(known_statuses)
n_unknown_status = int(unknown_status_mask.sum())
print(f"未知の watching_status 値を持つ行: {n_unknown_status:,} ({n_unknown_status/n_rows*100:.4f}%)")
if n_unknown_status > 0:
    print(df.loc[unknown_status_mask, "watching_status"].value_counts())


未知の watching_status 値を持つ行: 2 (0.0000%)
watching_status
5    2
Name: count, dtype: int64


## 4. `watched_episodes` の異常値チェック

`anime.csv` の総話数（`Episodes`）と突き合わせ、以下を検出する。
- 総話数超過（総話数が既知の作品のみ対象。`Episodes` が `Unknown` の作品は判定対象外）
- 負値
- 欠損（NaN）

※ `anime.csv` の完走率系の列（Watching/Completed/On-Hold/Dropped/Plan to Watch, Score-10〜Score-1）は
全MALユーザーの集計値であり `animelist.csv` の母集団と異なるため、ここでは総話数の突合のみに使用する
（チェック④の個人効果 σ の算出には使わない）。

In [7]:
anime = pd.read_csv(ANIME_PATH, usecols=["MAL_ID", "Episodes"])
anime["Episodes"] = pd.to_numeric(anime["Episodes"], errors="coerce")  # "Unknown" -> NaN
anime = anime.rename(columns={"MAL_ID": "anime_id", "Episodes": "total_episodes"})

df_m = df.merge(anime, on="anime_id", how="left")

n_null_watched = int(df_m["watched_episodes"].isna().sum())
n_negative = int((df_m["watched_episodes"] < 0).sum())
has_total = df_m["total_episodes"].notna()
n_overrun = int(((df_m["watched_episodes"] > df_m["total_episodes"]) & has_total).sum())
n_unknown_total = int((~has_total).sum())

anomaly_table = pd.DataFrame({
    "件数": [n_null_watched, n_negative, n_overrun, n_unknown_total],
    "割合(%)": [round(x / n_rows * 100, 4) for x in [n_null_watched, n_negative, n_overrun, n_unknown_total]],
}, index=["watched_episodes 欠損", "watched_episodes 負値", "総話数超過（総話数既知のみ）", "anime.csv側 総話数不明の行"])
anomaly_table


,件数,割合(%)
watched_episodes 欠損,0,0.000
watched_episodes 負値,0,0.000
総話数超過（総話数既知のみ）,18479,0.184
anime.csv側 総話数不明の行,89895,0.895


## 5. チェック① 離脱率

制約: 完走(2)と離脱(4)のみを母数とし、「完走＋離脱に占める離脱の割合」で評価する（クラス不均衡があるため、視聴予定などを含む全体比率ではなく、この定義を用いる）。

離脱の定義を2通りで算出する。
- **定義A**: 離脱(4) のみを「離脱」とする
- **定義B**: 離脱(4) + 保留(3) を「離脱」とする（保留の平均 watched_episodes は離脱寄りであるため、参考値として併記）

合格基準: 定義Aで **5%以上**。

In [8]:
n_completed = int((df["watching_status"] == 2).sum())
n_dropped = int((df["watching_status"] == 4).sum())
n_onhold = int((df["watching_status"] == 3).sum())

rate_A = n_dropped / (n_completed + n_dropped)
rate_B = (n_dropped + n_onhold) / (n_completed + n_dropped + n_onhold)

check1_pass = rate_A >= 0.05

print(f"完走: {n_completed:,} / 離脱(定義A): {n_dropped:,} / 保留: {n_onhold:,}")
print(f"定義A 離脱率: {rate_A*100:.2f}%")
print(f"定義B 離脱率: {rate_B*100:.2f}%")
print(f"チェック①合否（定義A, >=5%）: {'PASS' if check1_pass else 'FAIL'}")


完走: 6,306,027 / 離脱(定義A): 404,260 / 保留: 342,226
定義A 離脱率: 6.02%
定義B 離脱率: 10.58%
チェック①合否（定義A, >=5%）: PASS


## 6. チェック② 話数データの有効率

離脱ユーザーの `watched_episodes` が 0 より大きい割合。合格基準: 定義Aで **70%以上**。

In [9]:
dropped_df = df[df["watching_status"] == 4]
check2_A = (dropped_df["watched_episodes"] > 0).mean()

dropped_or_onhold_df = df[df["watching_status"].isin([3, 4])]
check2_B = (dropped_or_onhold_df["watched_episodes"] > 0).mean()

check2_pass = check2_A >= 0.70

print(f"定義A（離脱のみ, n={len(dropped_df):,}）watched_episodes>0 割合: {check2_A*100:.2f}%")
print(f"定義B（離脱+保留, n={len(dropped_or_onhold_df):,}）watched_episodes>0 割合: {check2_B*100:.2f}%")
print(f"チェック②合否（定義A, >=70%）: {'PASS' if check2_pass else 'FAIL'}")


定義A（離脱のみ, n=404,260）watched_episodes>0 割合: 76.21%
定義B（離脱+保留, n=746,486）watched_episodes>0 割合: 74.11%
チェック②合否（定義A, >=70%）: PASS


## 7. チェック③ `watching`（視聴中）の割合

15%未満であれば除外して問題ない（要件定義書 3.1）。事前確認（先頭500万行 4.8%）と一致するかをフルサンプルで検証する。

In [10]:
n_watching = int((df["watching_status"] == 1).sum())
watching_ratio = n_watching / n_rows
exclude_ok = watching_ratio < 0.15

print(f"watching(1) 件数: {n_watching:,} / 割合: {watching_ratio*100:.2f}%")
print(f"追加分析なしで除外可能（<15%）: {exclude_ok}")
print("→ 判定: Phase 1-2 では watching / plan_to_watch は除外する。除外率は上記の通り。")


watching(1) 件数: 478,018 / 割合: 4.76%
追加分析なしで除外可能（<15%）: True
→ 判定: Phase 1-2 では watching / plan_to_watch は除外する。除外率は上記の通り。


## 8. チェック④ 個人効果の存在【最重要】

作品ごとの平均完走率（**このサンプル内のユーザーのみから算出**。Phase 2 での「訓練ユーザーのみから算出」というリーク防止ルールと同じ考え方）を引いた残差の、ユーザー単位の平均の標準偏差 σ を求める。

合格基準: **σ > 0.08**。0.03〜0.08 はグレーゾーン。

定義A（完走 vs 離脱のみ）と定義B（完走 vs 離脱+保留）の両方で算出する。

In [11]:
def compute_sigma(df, statuses_as_dropped):
    sub = df[df["watching_status"].isin([2] + statuses_as_dropped)].copy()
    sub["completed_flag"] = (sub["watching_status"] == 2).astype(float)
    anime_avg = sub.groupby("anime_id")["completed_flag"].transform("mean")
    user_effect = (sub["completed_flag"] - anime_avg).groupby(sub["user_id"]).mean()
    return float(user_effect.std()), len(sub), sub["user_id"].nunique()

sigma_A, n_sub_A, n_users_A = compute_sigma(df, [4])
sigma_B, n_sub_B, n_users_B = compute_sigma(df, [3, 4])

def sigma_verdict(sigma):
    if sigma > 0.08:
        return "GO（σ > 0.08）"
    elif sigma >= 0.03:
        return "グレー（0.03〜0.08）"
    else:
        return "NO-GO（σ < 0.03）"

print(f"定義A（完走 vs 離脱）        : σ = {sigma_A:.4f}  (行数={n_sub_A:,}, ユーザー数={n_users_A:,}) → {sigma_verdict(sigma_A)}")
print(f"定義B（完走 vs 離脱+保留）    : σ = {sigma_B:.4f}  (行数={n_sub_B:,}, ユーザー数={n_users_B:,}) → {sigma_verdict(sigma_B)}")


定義A（完走 vs 離脱）        : σ = 0.0782  (行数=6,710,287, ユーザー数=29,557) → グレー（0.03〜0.08）
定義B（完走 vs 離脱+保留）    : σ = 0.1142  (行数=7,052,513, ユーザー数=29,600) → GO（σ > 0.08）


## 9. 追加集計: ユーザーあたり登録作品数の分布

In [12]:
reg_per_user = df.groupby("user_id").size()
reg_desc = reg_per_user.describe(percentiles=[0.25, 0.5, 0.75])
print(reg_desc)
print(f"\n中央値: {reg_per_user.median():.1f} / Q1: {reg_per_user.quantile(0.25):.1f} / Q3: {reg_per_user.quantile(0.75):.1f}")


count   30,000.000
mean       334.612
std        415.513
min          1.000
25%         95.000
50%        225.000
75%        435.000
max     11,938.000
dtype: float64

中央値: 225.0 / Q1: 95.0 / Q3: 435.0


## 10. 追加集計: 作品あたりの評価件数（登録件数）の分布

サンプル内（3万ユーザー）における作品ごとの登録件数。母集団（32万ユーザー）全体の件数ではない点に注意。

In [13]:
ratings_per_anime = df.groupby("anime_id").size()
ratings_desc = ratings_per_anime.describe(percentiles=[0.25, 0.5, 0.75])
print(ratings_desc)
print(f"\n対象作品数: {len(ratings_per_anime):,}")
print(f"中央値: {ratings_per_anime.median():.1f} / Q1: {ratings_per_anime.quantile(0.25):.1f} / Q3: {ratings_per_anime.quantile(0.75):.1f}")


count   17,299.000
mean       580.285
std      1,540.064
min          1.000
25%         12.000
50%         64.000
75%        346.500
max     21,866.000
dtype: float64

対象作品数: 17,299
中央値: 64.0 / Q1: 12.0 / Q3: 346.5


## 11. 完走 / 離脱のクラス比

In [14]:
ratio_A_completed = n_completed / (n_completed + n_dropped) * 100
ratio_A_dropped = n_dropped / (n_completed + n_dropped) * 100
ratio_B_completed = n_completed / (n_completed + n_dropped + n_onhold) * 100
ratio_B_dropped = (n_dropped + n_onhold) / (n_completed + n_dropped + n_onhold) * 100

print(f"定義A: 完走 {ratio_A_completed:.1f}% : 離脱 {ratio_A_dropped:.1f}%  (約 {round(ratio_A_completed/ratio_A_dropped)}:1)")
print(f"定義B: 完走 {ratio_B_completed:.1f}% : 離脱 {ratio_B_dropped:.1f}%  (約 {round(ratio_B_completed/ratio_B_dropped)}:1)")
print("\n→ クラス不均衡が大きいため、Phase 2 の評価は AUC / PR-AUC を用いる。Accuracy は使用しない。")


定義A: 完走 94.0% : 離脱 6.0%  (約 16:1)
定義B: 完走 89.4% : 離脱 10.6%  (約 8:1)

→ クラス不均衡が大きいため、Phase 2 の評価は AUC / PR-AUC を用いる。Accuracy は使用しない。


## 12. Go / No-Go 判定のまとめ

In [15]:
summary = pd.DataFrame([
    {"項目": "① 離脱率（定義A, 完走+離脱に占める割合）", "値": f"{rate_A*100:.2f}%", "基準": ">= 5%", "判定": "PASS" if check1_pass else "FAIL"},
    {"項目": "② watched_episodes>0 割合（定義A・離脱のみ）", "値": f"{check2_A*100:.2f}%", "基準": ">= 70%", "判定": "PASS" if check2_pass else "FAIL"},
    {"項目": "③ watching 割合", "値": f"{watching_ratio*100:.2f}%", "基準": "< 15% で除外可", "判定": "PASS" if exclude_ok else "要追加分析"},
    {"項目": "④ 個人効果 σ（定義A）", "値": f"{sigma_A:.4f}", "基準": "> 0.08", "判定": sigma_verdict(sigma_A)},
    {"項目": "④ 個人効果 σ（定義B）", "値": f"{sigma_B:.4f}", "基準": "> 0.08", "判定": sigma_verdict(sigma_B)},
])
summary


,項目,値,基準,判定
0,"① 離脱率（定義A, 完走+離脱に占める割合）",6.02%,>= 5%,PASS
1,② watched_episodes>0 割合（定義A・離脱のみ）,76.21%,>= 70%,PASS
2,③ watching 割合,4.76%,< 15% で除外可,PASS
3,④ 個人効果 σ（定義A）,0.0782,> 0.08,グレー（0.03〜0.08）
4,④ 個人効果 σ（定義B）,0.1142,> 0.08,GO（σ > 0.08）


In [16]:
go_strict_A = check1_pass and check2_pass and (sigma_A > 0.08)
go_with_B = check1_pass and check2_pass and (sigma_B > 0.08)

print(f"厳密な定義A（離脱=4のみ）での自動判定: {'GO' if go_strict_A else 'GO ではない（σがグレーゾーン、要判断）'}")
print(f"定義B（離脱=4+保留=3）を採用した場合の判定: {'GO' if go_with_B else 'NO-GO'}")
print()
print("結論: チェック①②は両定義で余裕を持って合格。チェック④のみ定義Aだと0.08をわずかに下回りグレーゾーン、")
print("定義Bを採用すると明確にGO水準（0.08超）となる。詳細な考察は reports/phase0_report.md を参照。")


厳密な定義A（離脱=4のみ）での自動判定: GO ではない（σがグレーゾーン、要判断）
定義B（離脱=4+保留=3）を採用した場合の判定: GO

結論: チェック①②は両定義で余裕を持って合格。チェック④のみ定義Aだと0.08をわずかに下回りグレーゾーン、
定義Bを採用すると明確にGO水準（0.08超）となる。詳細な考察は reports/phase0_report.md を参照。


---

**本ノートブックは Phase 0（データ検証）のみを実施するものであり、Phase 1 以降の実装には着手していない。**
判定結果と推奨事項は `reports/phase0_report.md` を参照。